<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Localhost and Service Binding

A network result depends on where a command runs. `localhost` and a listening address need careful interpretation. This notebook introduces the command contexts used in this LX, localhost, and service binding.

## Identify the current network context

On the base station, `dts` provides Duckietown-specific commands. To open a browser editor, open a base-station terminal, navigate to the base of this repository and run `dts code editor`. Its terminal, filesystem, processes, network interfaces, and `localhost` belong to a separate editor environment, not to the base station.

Figure 1 shows where to open a terminal in the browser editor. Commands entered there run in the editor environment, not directly in the browser host environment.

<figure id="figure-1" class="lx-figure">
  <img src="../assets/images/vs-code-editor-terminal.jpg" alt="Browser editor interface with its terminal opened" style="display:block; width:80%; margin:0 auto;">
  <figcaption>Figure 1: Opening a terminal in the browser editor.</figcaption>
</figure>

Keep a separate base-station terminal open for `dts`, base-station credentials, and checks that need the base station's network path to a Duckiedrone. Each context has its own loopback address, as summarized in [Table 1](#table-1).

<table id="table-1" class="lx-table">
  <caption>Table 1: Network contexts and their <code>localhost</code> targets.</caption>
  <thead>
    <tr>
      <th>Command runs in</th>
      <th><code>localhost</code> normally refers to</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>dts code editor</code> terminal</td>
      <td>The editor environment</td>
    </tr>
    <tr>
      <td>Base-station terminal</td>
      <td>The base station</td>
    </tr>
    <tr>
      <td>Shell on a physical Duckiedrone</td>
      <td>That Duckiedrone</td>
    </tr>
    <tr>
      <td>Shell on a virtual Duckiedrone</td>
      <td>That Duckiedrone</td>
    </tr>
    <tr>
      <td>Service shell</td>
      <td>That service environment</td>
    </tr>
  </tbody>
</table>

After opening a shell on a Duckiedrone, subsequent commands run in that Duckiedrone environment. A service shell within a Duckiedrone is another separate context, so `localhost` there refers to that service environment rather than to the Duckiedrone, base station, or editor.

For example, a browser running on the base station that opens `http://localhost:8080` targets a service on the base station. A program that runs in a shell on a Duckiedrone and requests the same URL targets that Duckiedrone instead. Opening an SSH shell does not move an already-running browser into the Duckiedrone context.

A request to `localhost` from a shell on a Duckiedrone does not automatically reach a base-station service. Use the documented address, hostname, and port when one component must reach another. Use `hostname` and `ip` to help establish where a command runs.

## Service binding

A service chooses a protocol, port, and one or more local addresses on which to accept new connections. This is called __binding__. It is separate from whether the service is running: a running service can be reachable in one network context and unavailable in another.

__Port forwarding__ is a forwarding rule that accepts connections at one address and port, then relays them to a service at another address and port. It does not change the service's listening address or bypass firewall and other access controls.

`nc`, also called netcat, can open one TCP connection to a specified address and port or listen for one TCP connection. A service bound to `127.0.0.1:PORT` or `[::1]:PORT`, where `PORT` is the documented port number for the service, accepts connections only from the same network context. A local test such as `nc 127.0.0.1 PORT` can succeed while a base-station connection to `DUCKIEDRONE_NAME.local:PORT` fails. That result alone does not establish the cause because name resolution, routing, and firewall rules still matter.

Some process listings show `0.0.0.0:PORT` for IPv4 or `[::]:PORT` for IPv6. These are wildcard listening addresses: the service accepts connections sent to matching IP addresses in its current context. Because a wildcard listener can expose a service to other computers on reachable networks, subject to firewall and service access controls, use one only when remote access is intended and authorized. Do not change a listening address merely to make a connection work. They are not destination addresses to type into a browser or SSH command. Whether an IPv6 wildcard listener also accepts IPv4 connections depends on the service and its configuration.

When a documented service cannot be reached, verify the intended hostname or address, protocol, and port before changing a binding. In this LX, inspect rather than modify service or network settings.

Hypertext Transfer Protocol (HTTP) is a request-response application protocol commonly used for web resources. The activity below uses it only to create and inspect a local listener.

`python3 -m http.server 8000` starts a temporary Python HTTP server that serves the current directory until you stop it with `Ctrl-C`. The `--bind 127.0.0.1` option makes that server listen only on the current context's IPv4 loopback address, so it is not exposed to another computer.

[Notebook 18](./18-network-protocols-and-quality.ipynb) introduced `ss`, which can report the resulting TCP listener. In `ss -ltn 'sport = :8000'`, the `sport = :8000` filter limits the listing to listeners whose local port is `8000`. `curl --head` sends an HTTP request for headers only; it does not download the response body.

### Try it

Use two terminals in the same local Linux context. In the first, start a temporary loopback-only web server:

```bash
python3 -m http.server 8000 --bind 127.0.0.1
```

In the second, inspect and contact that listener:

```bash
ss -ltn 'sport = :8000'
curl --head http://127.0.0.1:8000
```

Confirm that the listener is bound to loopback, then press `Ctrl-C` in the first terminal to stop it. Do not change the binding to expose the temporary server to another computer.

<details>
<summary>Check your result</summary>

The request succeeds because both commands run in the same context and the service accepts `127.0.0.1:8000`. A Duckiedrone, base station, or browser editor has its own `localhost`, so that result does not make the server reachable from another context.

</details>

## Further reading

The Linux [`ip(7)` manual](https://man7.org/linux/man-pages/man7/ip.7.html) describes local IPv4 socket addresses and binding.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
